[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/filippolonghi/medleydb-mert-instrument-classification/blob/main/notebooks/00_dataset_exploration_medleydb.ipynb)

# 00 ? Dataset exploration: MedleyDB isolated stems

## Research question

What data do we actually have available for isolated-stem instrument classification on MedleyDB?

## Approach

This notebook reads the generated stem index and subset reports, then creates report-ready tables and plots. It does not train any model.

## What is fixed

The task remains isolated-stem classification, with complete 5-second segment windows and track-level train/validation/test splitting.

## What is varied

We compare the two label granularities (`coarse_family`, `medleydb_instrument`) and the two subset profiles (`debug`, `largest_balanced`).

## Expected interpretation

The goal is to justify the final protocol before showing model results. The final report should prioritize `largest_balanced + medleydb_instrument` when enough data is available.

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

# Edit these three variables when moving between local runs and Colab/Drive.
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/content/medleydb-mert-instrument-classification"))
MEDLEYDB_ROOT = Path(os.environ.get("MEDLEYDB_ROOT", "/content/drive/MyDrive/medleydb_mert_project/MedleyDB"))
RUN_ROOT = Path(os.environ.get("RUN_ROOT", "/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1"))

SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"

RESULTS_DIR = RUN_ROOT / "results"
CHECKPOINTS_DIR = RUN_ROOT / "checkpoints"
METADATA_DIR = RUN_ROOT / "data" / "metadata"
CACHE_ROOT = RUN_ROOT / "data" / "cache"
REPORTS_DIR = RUN_ROOT / "data" / "reports"
for path in [RESULTS_DIR, CHECKPOINTS_DIR, METADATA_DIR, CACHE_ROOT, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)
print("Run root:", RUN_ROOT)
print("Subset profile:", SUBSET_PROFILE)
print("Label granularity:", LABEL_GRANULARITY)

def q(path):
    """Quote paths safely for copied shell commands, including Drive paths with spaces."""
    return shlex.quote(str(path))

def run(cmd, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check)


In [ ]:
from pathlib import Path
import os
import sys

# Force the same final paths used in pipeline_medleydb.ipynb
PROJECT_ROOT = Path("/content/medleydb-mert-instrument-classification")
MEDLEYDB_ROOT = Path("/content/drive/MyDrive/medleydb-mert-instrument-classification/MedleyDB")
RUN_ROOT = Path("/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1")

SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"

RESULTS_DIR = RUN_ROOT / "results"
CHECKPOINTS_DIR = RUN_ROOT / "checkpoints"
METADATA_DIR = RUN_ROOT / "data" / "metadata"
CACHE_ROOT = RUN_ROOT / "data" / "cache"
REPORTS_DIR = RUN_ROOT / "data" / "reports"

for path in [RESULTS_DIR, CHECKPOINTS_DIR, METADATA_DIR, CACHE_ROOT, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)

if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())

print("MEDLEYDB_ROOT:", MEDLEYDB_ROOT)
print("MEDLEYDB_ROOT exists:", MEDLEYDB_ROOT.is_dir())
print("MEDLEYDB_ROOT/Audio exists:", (MEDLEYDB_ROOT / "Audio").is_dir())

print("RUN_ROOT:", RUN_ROOT)
print("RUN_ROOT exists:", RUN_ROOT.is_dir())

print("METADATA_DIR:", METADATA_DIR)
print("METADATA_DIR exists:", METADATA_DIR.is_dir())

print("RESULTS_DIR:", RESULTS_DIR)
print("RESULTS_DIR exists:", RESULTS_DIR.is_dir())

print("Current working directory:", Path.cwd())

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
from google.colab import drive, userdata

# 1. Mount Drive
drive.mount("/content/drive")

# 2. Final paths
PROJECT_ROOT = Path("/content/medleydb-mert-instrument-classification")
MEDLEYDB_ROOT = Path("/content/drive/MyDrive/medleydb-mert-instrument-classification/MedleyDB")
RUN_ROOT = Path("/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1")

SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"

RESULTS_DIR = RUN_ROOT / "results"
CHECKPOINTS_DIR = RUN_ROOT / "checkpoints"
METADATA_DIR = RUN_ROOT / "data" / "metadata"
CACHE_ROOT = RUN_ROOT / "data" / "cache"
REPORTS_DIR = RUN_ROOT / "data" / "reports"

# 3. Clone/pull repository
token = userdata.get("GITHUB_TOKEN")
if token is None:
    raise RuntimeError("GITHUB_TOKEN not found in Colab Secrets.")

repo_url = "https://github.com/Frabbandera/medleydb-mert-instrument-classification.git"

if PROJECT_ROOT.exists():
    print("Repository already exists. Pulling latest changes...")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
else:
    print("Repository not found. Cloning...")
    subprocess.run(["git", "clone", repo_url, str(PROJECT_ROOT)], check=True)

# 4. Install requirements
os.chdir(PROJECT_ROOT)
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# 5. Export environment variables
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["HF_HOME"] = str(RUN_ROOT / "huggingface")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("\nSETUP COMPLETE")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())

print("MEDLEYDB_ROOT:", MEDLEYDB_ROOT)
print("MEDLEYDB_ROOT exists:", MEDLEYDB_ROOT.is_dir())
print("MEDLEYDB_ROOT/Audio exists:", (MEDLEYDB_ROOT / "Audio").is_dir())

print("RUN_ROOT:", RUN_ROOT)
print("RUN_ROOT exists:", RUN_ROOT.is_dir())

print("Current working directory:", Path.cwd())

In [ ]:
from pathlib import Path
import os

paths_to_check = [
    Path("/content/drive"),
    Path("/content/drive/MyDrive"),
    Path("/content/drive/MyDrive/medleydb_mert_project"),
    Path("/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1"),
    Path("/content/drive/MyDrive/medleydb-mert-instrument-classification"),
    Path("/content/drive/MyDrive/medleydb-mert-instrument-classification/MedleyDB"),
    Path("/content/drive/MyDrive/medleydb-mert-instrument-classification/MedleyDB/Audio"),
]

for p in paths_to_check:
    print(p, "->", p.exists(), "| is_dir:", p.is_dir(), "| is_symlink:", p.is_symlink())

print("\nTop-level /content/drive:")
if Path("/content/drive").exists():
    for p in sorted(Path("/content/drive").iterdir()):
        print(" ", p)

print("\nTop-level MyDrive:")
if Path("/content/drive/MyDrive").exists():
    for p in sorted(Path("/content/drive/MyDrive").iterdir()):
        print(" ", p)

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
from google.colab import userdata

# Do NOT call drive.mount() here: Drive is already visible.

PROJECT_ROOT = Path("/content/medleydb-mert-instrument-classification")
RUN_ROOT = Path("/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1")

# In this runtime the old MedleyDB shortcut is not visible.
# Keep the expected path for now, but we will not rely on it for notebook 00 except optional audio examples.
MEDLEYDB_ROOT = Path("/content/drive/MyDrive/medleydb-mert-instrument-classification/MedleyDB")

SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"

RESULTS_DIR = RUN_ROOT / "results"
CHECKPOINTS_DIR = RUN_ROOT / "checkpoints"
METADATA_DIR = RUN_ROOT / "data" / "metadata"
CACHE_ROOT = RUN_ROOT / "data" / "cache"
REPORTS_DIR = RUN_ROOT / "data" / "reports"

# Clone/pull repo
token = userdata.get("GITHUB_TOKEN")
if token is None:
    raise RuntimeError("GITHUB_TOKEN not found in Colab Secrets.")

repo_url = "https://github.com/Frabbandera/medleydb-mert-instrument-classification.git"

if PROJECT_ROOT.exists():
    print("Repository already exists. Pulling latest changes...")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
else:
    print("Repository not found. Cloning...")
    subprocess.run(["git", "clone", repo_url, str(PROJECT_ROOT)], check=True)

# Install requirements
os.chdir(PROJECT_ROOT)
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Export environment variables
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Create output folders if missing
for path in [RESULTS_DIR, CHECKPOINTS_DIR, METADATA_DIR, CACHE_ROOT, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("\nSETUP COMPLETE")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())

print("RUN_ROOT:", RUN_ROOT)
print("RUN_ROOT exists:", RUN_ROOT.is_dir())

print("METADATA_DIR:", METADATA_DIR)
print("METADATA_DIR exists:", METADATA_DIR.is_dir())

print("RESULTS_DIR:", RESULTS_DIR)
print("RESULTS_DIR exists:", RESULTS_DIR.is_dir())

print("MEDLEYDB_ROOT:", MEDLEYDB_ROOT)
print("MEDLEYDB_ROOT exists:", MEDLEYDB_ROOT.is_dir())
print("MEDLEYDB_ROOT/Audio exists:", (MEDLEYDB_ROOT / "Audio").is_dir())

print("Current working directory:", Path.cwd())

In [ ]:
from pathlib import Path

required_pipeline_outputs = [
    METADATA_DIR / "stem_index.csv",
    METADATA_DIR / "subset_largest_balanced_medleydb_instrument.csv",
    METADATA_DIR / "labels_largest_balanced_medleydb_instrument_label_to_id.json",
    REPORTS_DIR / "data_health_report.md",
    REPORTS_DIR / "subset_report_largest_balanced_medleydb_instrument.md",
    RESULTS_DIR / "experiment_registry.csv",
]

for path in required_pipeline_outputs:
    print(path.name, "->", path.is_file())

In [ ]:
from pathlib import Path
import subprocess

print("Mount information:")
mount_info = subprocess.run(
    "mount | grep -E 'content/drive|drivefs|google' || true",
    shell=True,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
print(mount_info.stdout if mount_info.stdout.strip() else "No Google Drive mount detected.")

print("\n/content/drive exists:", Path("/content/drive").exists())
print("/content/drive is_dir:", Path("/content/drive").is_dir())

if Path("/content/drive").exists():
    print("\nTop-level /content/drive:")
    for p in sorted(Path("/content/drive").iterdir()):
        print(" ", p)

print("\nTop-level /content:")
for p in sorted(Path("/content").iterdir()):
    if p.name.startswith("drive"):
        print(" ", p)

In [ ]:
from pathlib import Path
import shutil
from google.colab import drive

drive_path = Path("/content/drive")
backup_path = Path("/content/drive_local_backup_before_real_mount")

# If /content/drive exists but is not a real mounted Drive, move it away safely.
if drive_path.exists():
    if backup_path.exists():
        shutil.rmtree(backup_path)
    print("Moving existing local /content/drive to:", backup_path)
    shutil.move(str(drive_path), str(backup_path))

# Recreate empty mountpoint
drive_path.mkdir(parents=True, exist_ok=True)

# Mount real Google Drive
drive.mount("/content/drive")

print("\nMounted Drive content:")
for p in sorted(Path("/content/drive/MyDrive").iterdir()):
    print(" ", p)

In [ ]:
from pathlib import Path

paths_to_check = [
    Path("/content/drive/MyDrive/medleydb_mert_project"),
    Path("/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1"),
    Path("/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1/data/metadata/stem_index.csv"),
    Path("/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1/data/metadata/subset_largest_balanced_medleydb_instrument.csv"),
    Path("/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1/results/experiment_registry.csv"),
    Path("/content/drive/MyDrive/medleydb-mert-instrument-classification"),
    Path("/content/drive/MyDrive/medleydb-mert-instrument-classification/MedleyDB"),
    Path("/content/drive/MyDrive/medleydb-mert-instrument-classification/MedleyDB/Audio"),
]

for p in paths_to_check:
    print(p, "->", p.exists(), "| is_dir:", p.is_dir())

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path("/content/medleydb-mert-instrument-classification")
MEDLEYDB_ROOT = Path("/content/drive/MyDrive/medleydb-mert-instrument-classification/MedleyDB")
RUN_ROOT = Path("/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1")

SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"

RESULTS_DIR = RUN_ROOT / "results"
CHECKPOINTS_DIR = RUN_ROOT / "checkpoints"
METADATA_DIR = RUN_ROOT / "data" / "metadata"
CACHE_ROOT = RUN_ROOT / "data" / "cache"
REPORTS_DIR = RUN_ROOT / "data" / "reports"

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)

if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())

print("MEDLEYDB_ROOT:", MEDLEYDB_ROOT)
print("MEDLEYDB_ROOT exists:", MEDLEYDB_ROOT.is_dir())
print("MEDLEYDB_ROOT/Audio exists:", (MEDLEYDB_ROOT / "Audio").is_dir())

print("RUN_ROOT:", RUN_ROOT)
print("RUN_ROOT exists:", RUN_ROOT.is_dir())

print("METADATA_DIR:", METADATA_DIR)
print("METADATA_DIR exists:", METADATA_DIR.is_dir())

print("RESULTS_DIR:", RESULTS_DIR)
print("RESULTS_DIR exists:", RESULTS_DIR.is_dir())

print("Current working directory:", Path.cwd())

In [ ]:
from pathlib import Path

required_pipeline_outputs = [
    METADATA_DIR / "stem_index.csv",
    METADATA_DIR / "subset_largest_balanced_medleydb_instrument.csv",
    METADATA_DIR / "labels_largest_balanced_medleydb_instrument_label_to_id.json",
    REPORTS_DIR / "data_health_report.md",
    REPORTS_DIR / "subset_report_largest_balanced_medleydb_instrument.md",
    RESULTS_DIR / "experiment_registry.csv",
]

for path in required_pipeline_outputs:
    print(path.name, "->", path.is_file())

## What you can change

- `PROJECT_ROOT`, `RUN_ROOT`, and `MEDLEYDB_ROOT` for local vs Colab/Drive paths.
- Config path variables such as `EXPERIMENT_CONFIG` or `MIXTURE_CONFIG`.
- `debug` is for quick smoke tests; `largest_balanced_medleydb_instrument` is the final main dataset.
- Polyphonic modes: `synthetic_random` is controlled overlap, `same_song_reconstructed` is realistic stem reconstruction, and `original_full_mix` is real mix evaluation.


## Load metadata

Run Stage 1 and Stage 2 first if the CSV files are missing. The pipeline notebook creates these files under `RUN_ROOT`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

index_path = METADATA_DIR / "stem_index.csv"
if not index_path.exists():
    print("Missing stem_index.csv. Run Stage 1 in pipeline_medleydb.ipynb first.")
else:
    index = pd.read_csv(index_path)
    print(index.shape)
    display(index.head())

In [ ]:
print("index loaded:", "index" in globals())

if "index" in globals():
    print("index shape:", index.shape)
    print("columns:", list(index.columns))
    print("valid stems:", int(index["valid"].astype(bool).sum()))
    print("invalid stems:", int((~index["valid"].astype(bool)).sum()))
    print("unique tracks:", index["track_id"].nunique())
    print("unique medleydb instruments:", index["medleydb_instrument_label"].nunique())

## Dataset health and technical properties

These plots summarize valid/invalid stems, sample rates, channels, and duration. They are saved to `RUN_ROOT/results/dataset_exploration/`.

In [ ]:
plot_dir = RESULTS_DIR / "dataset_exploration"
plot_dir.mkdir(parents=True, exist_ok=True)
if 'index' in globals():
    summary = pd.DataFrame({
        "count": [len(index), int(index["valid"].astype(bool).sum()), int((~index["valid"].astype(bool)).sum())]
    }, index=["indexed_stems", "valid_stems", "invalid_stems"])
    display(summary)
    summary.to_csv(plot_dir / "stem_health_summary.csv")

    for column, title in [("sample_rate", "Sample-rate distribution"), ("channels", "Channel distribution")]:
        if column in index:
            ax = index[column].value_counts(dropna=False).sort_index().plot(kind="bar", title=title)
            ax.set_xlabel(column)
            ax.set_ylabel("stems")
            plt.tight_layout()
            plt.savefig(plot_dir / f"{column}_distribution.png", dpi=160)
            plt.show()

    if "duration_seconds" in index:
        ax = index["duration_seconds"].dropna().plot(kind="hist", bins=40, title="Stem duration distribution")
        ax.set_xlabel("duration [s]")
        plt.tight_layout()
        plt.savefig(plot_dir / "duration_distribution.png", dpi=160)
        plt.show()

In [ ]:
plot_dir = RESULTS_DIR / "dataset_exploration"
plot_dir.mkdir(parents=True, exist_ok=True)

if 'index' in globals():
    summary = pd.DataFrame({
        "count": [
            len(index),
            int(index["valid"].astype(bool).sum()),
            int((~index["valid"].astype(bool)).sum())
        ]
    }, index=["indexed_stems", "valid_stems", "invalid_stems"])

    display(summary)
    summary.to_csv(plot_dir / "stem_health_summary.csv")

    for column, title in [
        ("sample_rate", "Sample-rate distribution"),
        ("num_channels", "Channel distribution")
    ]:
        if column in index:
            ax = index[column].value_counts(dropna=False).sort_index().plot(
                kind="bar",
                title=title
            )
            ax.set_xlabel(column)
            ax.set_ylabel("stems")
            plt.tight_layout()
            plt.savefig(plot_dir / f"{column}_distribution.png", dpi=160)
            plt.show()

    if "duration_seconds" in index:
        ax = index["duration_seconds"].dropna().plot(
            kind="hist",
            bins=40,
            title="Stem duration distribution"
        )
        ax.set_xlabel("duration [s]")
        plt.tight_layout()
        plt.savefig(plot_dir / "duration_distribution.png", dpi=160)
        plt.show()
else:
    print("index is not loaded. Run the metadata loading cell first.")

In [ ]:
from pathlib import Path
import pandas as pd

plot_dir = RESULTS_DIR / "dataset_exploration"

expected_outputs = [
    plot_dir / "stem_health_summary.csv",
    plot_dir / "sample_rate_distribution.png",
    plot_dir / "num_channels_distribution.png",
    plot_dir / "duration_distribution.png",
]

for path in expected_outputs:
    print(path.name, "->", path.is_file())

summary_path = plot_dir / "stem_health_summary.csv"
if summary_path.is_file():
    summary = pd.read_csv(summary_path, index_col=0)
    display(summary)

## Label analysis

`medleydb_instrument` is the instrument-level protocol: it keeps original MedleyDB labels with light formatting normalization. `coarse_family` is a controlled family-level protocol.

In [ ]:
if 'index' in globals():
    label_cols = [c for c in ["raw_instrument_label", "medleydb_instrument_label", "coarse_label"] if c in index]
    for column in label_cols:
        counts = index[column].value_counts().rename_axis(column).reset_index(name="stems")
        display(counts.head(25))
        counts.to_csv(plot_dir / f"{column}_counts.csv", index=False)
        ax = counts.head(20).plot(kind="bar", x=column, y="stems", legend=False, title=f"Top {column}")
        ax.set_ylabel("stems")
        plt.tight_layout()
        plt.savefig(plot_dir / f"{column}_top20.png", dpi=160)
        plt.show()

In [ ]:
from pathlib import Path
import pandas as pd

plot_dir = RESULTS_DIR / "dataset_exploration"

expected_outputs = [
    plot_dir / "raw_instrument_label_counts.csv",
    plot_dir / "medleydb_instrument_label_counts.csv",
    plot_dir / "coarse_label_counts.csv",
    plot_dir / "raw_instrument_label_top20.png",
    plot_dir / "medleydb_instrument_label_top20.png",
    plot_dir / "coarse_label_top20.png",
]

for path in expected_outputs:
    print(path.name, "->", path.is_file())

print("\nTop MedleyDB instrument labels:")
medley_counts = pd.read_csv(plot_dir / "medleydb_instrument_label_counts.csv")
display(medley_counts.head(15))

print("\nTop coarse labels:")
coarse_counts = pd.read_csv(plot_dir / "coarse_label_counts.csv")
display(coarse_counts.head(15))

## Subset construction checks

This section looks for the four official subset files and verifies that train/validation/test tracks are disjoint.

In [ ]:
subset_files = [
    "subset_debug_coarse_family.csv",
    "subset_debug_medleydb_instrument.csv",
    "subset_largest_balanced_coarse_family.csv",
    "subset_largest_balanced_medleydb_instrument.csv",
]
rows = []
for file_name in subset_files:
    path = METADATA_DIR / file_name
    if not path.exists():
        rows.append({"subset": file_name, "status": "missing"})
        continue
    subset = pd.read_csv(path)
    overlap = {}
    split_tracks = {s: set(subset.loc[subset.split == s, "track_id"].astype(str)) for s in ["train", "val", "test"]}
    overlap["train_val_overlap"] = len(split_tracks["train"] & split_tracks["val"])
    overlap["train_test_overlap"] = len(split_tracks["train"] & split_tracks["test"])
    overlap["val_test_overlap"] = len(split_tracks["val"] & split_tracks["test"])
    rows.append({"subset": file_name, "segments": len(subset), "classes": subset["label_name"].nunique(), **overlap})
    display(subset.groupby(["label_name", "split"]).size().unstack(fill_value=0))
summary = pd.DataFrame(rows)
display(summary)
summary.to_csv(plot_dir / "subset_summary.csv", index=False)

In [ ]:
from pathlib import Path
import pandas as pd

subset_summary_path = plot_dir / "subset_summary.csv"

print("subset_summary.csv exists:", subset_summary_path.is_file())

if subset_summary_path.is_file():
    subset_summary = pd.read_csv(subset_summary_path)
    display(subset_summary)

final_subset_row = subset_summary[
    subset_summary["subset"] == "subset_largest_balanced_medleydb_instrument.csv"
]

print("\nFinal subset row exists:", len(final_subset_row) == 1)

if len(final_subset_row) == 1:
    row = final_subset_row.iloc[0]
    print("segments:", row.get("segments"))
    print("classes:", row.get("classes"))
    print("train_val_overlap:", row.get("train_val_overlap"))
    print("train_test_overlap:", row.get("train_test_overlap"))
    print("val_test_overlap:", row.get("val_test_overlap"))

subset_path = METADATA_DIR / "subset_largest_balanced_medleydb_instrument.csv"
subset = pd.read_csv(subset_path)

print("\nFinal subset shape:", subset.shape)
print("\nClass x split:")
display(pd.crosstab(subset["label_name"], subset["split"]))

print("\nSplit counts:")
display(subset["split"].value_counts().sort_index())

print("\nClass counts:")
display(subset["label_name"].value_counts().sort_index())

## Audio examples

This optional cell plots one waveform and spectrogram from the selected subset. It helps explain that an isolated stem is one source/stem, not a full polyphonic mixture.

In [ ]:
# Optional: run only after the dataset is mounted and a subset CSV exists.
import numpy as np
try:
    import librosa
    import librosa.display
    from src.data.audio_io import load_audio_segment
    subset_path = METADATA_DIR / "subset_largest_balanced_medleydb_instrument.csv"
    if subset_path.exists():
        subset = pd.read_csv(subset_path)
        example = subset.iloc[0]
        audio_path = MEDLEYDB_ROOT / example.audio_path
        waveform, sr = load_audio_segment(audio_path, float(example.start_seconds), float(example.duration_seconds), normalize=True)
        plt.figure(figsize=(10, 3))
        librosa.display.waveshow(waveform.numpy(), sr=sr)
        plt.title(f"Waveform: {example.label_name}")
        plt.tight_layout()
        plt.savefig(plot_dir / "example_waveform.png", dpi=160)
        plt.show()

        S = librosa.amplitude_to_db(np.abs(librosa.stft(waveform.numpy())), ref=np.max)
        plt.figure(figsize=(10, 4))
        librosa.display.specshow(S, sr=sr, x_axis="time", y_axis="hz")
        plt.colorbar(format="%+2.0f dB")
        plt.title(f"Spectrogram: {example.label_name}")
        plt.tight_layout()
        plt.savefig(plot_dir / "example_spectrogram.png", dpi=160)
        plt.show()
    else:
        print("Subset file not found yet.")
except Exception as exc:
    print("Audio example skipped:", exc)